# Langgraph Agent

In [ ]:
import os
import logging
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

print("Environment loaded")

In [ ]:
%run mcp_layer.ipynb

print("\n" + "="*80)
print("  TOOLS LOADED")
print("="*80)
print("✅ ZoningLawTool")
print("✅ VisionTool")
print("✅ FMVTool")
print("✅ WalkabilityTool")
print("✅ MarketExpertTool")
print("="*80)

In [ ]:
print("Initializing tools...\n")

# Initialize each tool
zoning_tool = ZoningLawTool(
    index_name="orlando-zoning-index",
    namespace="__default__"
)
print("✅ Zoning Law Tool ready")

vision_tool = VisionTool(
    index_name="orlando-zoning-index",
    namespace="property-damage"
)
print("✅ Vision Tool ready")

fmv_tool = FMVTool()
print("✅ FMV Tool ready")

walkability_tool = WalkabilityTool()
print("✅ Walkability Tool ready")

market_expert_tool = MarketExpertTool(
    model_config_path="../output/orlando_mkt_data_expert_metadata.json"
)
print("✅ Market Expert Tool ready")

print("\n🎉 All tools initialized successfully!")

### Agent Class

In [ ]:
class OrlandoRealEstateAgent:
    """
    Simple agent that orchestrates all 5 tools to analyze Orlando properties.
    """
    
    def __init__(self, zoning, vision, fmv, walkability, market_expert):
        self.zoning = zoning
        self.vision = vision
        self.fmv = fmv
        self.walkability = walkability
        self.market_expert = market_expert
        
        print("🤖 Orlando Real Estate Agent initialized")
    
    def analyze_property(
        self,
        latitude: float,
        longitude: float,
        land_sqft: float,
        living_area: float,
        age: int,
        structure_quality: float,
        SPEC_FEAT_VAL: float = 10000,
        rail_dist: float = 5000,
        ocean_dist: float = 55000,
        water_dist: float = 1500,
        center_dist: float = 40000,
        subcenter_dist: float = 8000,
        highway_dist: float = 3000,
        month_sold: int = 6,
        avno60plus: int = 0,
        image_path: str = None,
        zoning_question: str = None
    ):
        """
        Comprehensive property analysis using all available tools.
        
        Returns a structured analysis report.
        """
        print("\n" + "="*80)
        print("  🏠 ORLANDO PROPERTY ANALYSIS")
        print("="*80)
        print(f"\n📍 Location: ({latitude}, {longitude})")
        print(f"📐 Size: {living_area} sq ft living, {land_sqft} sq ft land")
        print(f"📅 Age: {age} years")
        print(f"⭐ Quality: {structure_quality}/5.0")
        print("\n" + "-"*80)
        
        results = {}
        
        # Step 1: FMV Prediction
        print("\n💰 Step 1/5: Predicting Fair Market Value...")
        try:
            fmv_result = self.fmv(
                latitude=latitude,
                longitude=longitude,
                land_sqft=land_sqft,
                living_area=living_area,
                age=age,
                structure_quality=structure_quality,
                SPEC_FEAT_VAL=SPEC_FEAT_VAL,
                rail_dist=rail_dist,
                ocean_dist=ocean_dist,
                water_dist=water_dist,
                center_dist=center_dist,
                subcenter_dist=subcenter_dist,
                highway_dist=highway_dist,
                month_sold=month_sold,
                avno60plus=avno60plus
            )
            results['fmv'] = fmv_result
            print(f"   ✅ Predicted FMV: ${fmv_result.predicted_fmv:,.2f}")
        except Exception as e:
            print(f"   ❌ FMV prediction failed: {e}")
            results['fmv'] = None
        
        # Step 2: Walkability Assessment
        print("\n🚶 Step 2/5: Assessing Walkability...")
        try:
            walk_result = self.walkability(
                latitude=latitude,
                longitude=longitude,
                radius_miles=1.0
            )
            results['walkability'] = walk_result
            print(f"   ✅ Walkability Score: {walk_result.walkability_score}/100 ({walk_result.walkability_interpretation})")
        except Exception as e:
            print(f"   ❌ Walkability assessment failed: {e}")
            results['walkability'] = None
        
        # Step 3: Zoning Laws (if question provided)
        if zoning_question:
            print(f"\n🏛️ Step 3/5: Checking Zoning Laws...")
            try:
                zoning_result = self.zoning(query=zoning_question)
                results['zoning'] = zoning_result
                print(f"   ✅ Found {zoning_result.chunks_retrieved} relevant zoning regulations")
            except Exception as e:
                print(f"   ❌ Zoning query failed: {e}")
                results['zoning'] = None
        else:
            print("\n🏛️ Step 3/5: Zoning check skipped (no question provided)")
            results['zoning'] = None
        
        # Step 4: Property Damage Analysis (if image provided)
        if image_path:
            print(f"\n📸 Step 4/5: Analyzing Property Images...")
            try:
                vision_result = self.vision(
                    image_path=image_path,
                    mode="analyze"
                )
                results['vision'] = vision_result
                print(f"   ✅ Damage: {vision_result.damage_type} (Severity: {vision_result.severity}/10)")
            except Exception as e:
                print(f"   ❌ Image analysis failed: {e}")
                results['vision'] = None
        else:
            print("\n📸 Step 4/5: Image analysis skipped (no image provided)")
            results['vision'] = None
        
        # Step 5: Market Expert Insights
        print(f"\n🧠 Step 5/5: Consulting Market Expert...")
        try:
            # Build context from previous results
            context_parts = []
            
            if results['fmv']:
                context_parts.append(f"predicted FMV of ${results['fmv'].predicted_fmv:,.0f}")
            
            if results['walkability']:
                context_parts.append(f"walkability score of {results['walkability'].walkability_score}/100")
            
            if results['vision']:
                context_parts.append(f"{results['vision'].damage_type} with severity {results['vision'].severity}/10")
            
            context = ", ".join(context_parts) if context_parts else "standard property"
            
            expert_query = f"I'm looking at a {age}-year-old property in Orlando with {context}. What should I know about this property and area?"
            
            expert_result = self.market_expert(query=expert_query, max_tokens=500)
            results['market_expert'] = expert_result
            print(f"   ✅ Expert insights generated ({expert_result.tokens_used} tokens)")
        except Exception as e:
            print(f"   ❌ Market expert query failed: {e}")
            results['market_expert'] = None
        
        print("\n" + "="*80)
        print("  ✅ ANALYSIS COMPLETE")
        print("="*80)
        
        return results
    
    def print_report(self, results):
        """Print a formatted analysis report"""
        print("\n" + "┏" + "━"*78 + "┓")
        print("┃" + " COMPREHENSIVE PROPERTY ANALYSIS REPORT ".center(78) + "┃")
        print("┗" + "━"*78 + "┛")
        
        # FMV Section
        if results.get('fmv'):
            fmv = results['fmv']
            print("\n💰 FAIR MARKET VALUE")
            print("─"*80)
            print(f"   Predicted Value: ${fmv.predicted_fmv:,.2f}")
            print()
        
        # Walkability Section
        if results.get('walkability'):
            walk = results['walkability']
            print("\n🚶 WALKABILITY ANALYSIS")
            print("─"*80)
            print(f"   Score: {walk.walkability_score}/100 ({walk.walkability_interpretation})")
            print(f"   {walk.summary}")
            print(f"\n   Nearby Amenities:")
            print(f"   • Grocery Stores: {walk.groceries.count_within_1mile} within 1 mile")
            print(f"   • Restaurants: {walk.restaurants.count_within_1mile} within 1 mile")
            print(f"   • Parks: {walk.parks.count_within_1mile} within 1 mile")
            print(f"   • Transit: {walk.transit.count_within_1mile} within 1 mile")
            print()
        
        # Zoning Section
        if results.get('zoning'):
            zoning = results['zoning']
            print("\n🏛️ ZONING INFORMATION")
            print("─"*80)
            print(f"   Query: {zoning.query}")
            print(f"\n   Answer:")
            for line in zoning.answer.split('\n'):
                print(f"   {line}")
            print()
        
        # Damage Section
        if results.get('vision'):
            vision = results['vision']
            print("\n📸 PROPERTY CONDITION")
            print("─"*80)
            print(f"   Damage Type: {vision.damage_type}")
            print(f"   Severity: {vision.severity}/10")
            print(f"   Urgency: {vision.urgency}")
            print(f"   Affected System: {vision.affected_system}")
            print(f"   Estimated Repair Cost: {vision.estimated_repair_cost}")
            print(f"\n   Summary: {vision.maintenance_summary}")
            print()
        
        # Market Expert Section
        if results.get('market_expert'):
            expert = results['market_expert']
            print("\n🧠 MARKET EXPERT INSIGHTS")
            print("─"*80)
            for line in expert.response.split('\n'):
                print(f"   {line}")
            print()
        
        print("\n" + "┏" + "━"*78 + "┓")
        print("┃" + " END OF REPORT ".center(78) + "┃")
        print("┗" + "━"*78 + "┛\n")


# Initialize the agent
agent = OrlandoRealEstateAgent(
    zoning=zoning_tool,
    vision=vision_tool,
    fmv=fmv_tool,
    walkability=walkability_tool,
    market_expert=market_expert_tool
)

### Example 1 -> Basic Property Analysis (No Image, No Zoning)

In [ ]:
print("\n" + "🏠 EXAMPLE 1: BASIC PROPERTY ANALYSIS" + "\n")

results = agent.analyze_property(
    # Location - Downtown Orlando
    latitude=28.5383,
    longitude=-81.3792,
    
    # Property characteristics
    land_sqft=8500,
    living_area=2000,
    age=15,
    structure_quality=4.0,
    SPEC_FEAT_VAL=10000,
    
    # Distances
    rail_dist=5000,
    ocean_dist=55000,
    water_dist=1500,
    center_dist=5000,  # Close to downtown
    subcenter_dist=8000,
    highway_dist=3000
)

# Print formatted report
agent.print_report(results)

### Example 2 - Property with Zoning Question

In [ ]:
print("\n" + "🏠 EXAMPLE 2: PROPERTY WITH ZONING INQUIRY" + "\n")

results = agent.analyze_property(
    # Location - Suburban Orlando
    latitude=28.5200,
    longitude=-81.2500,
    
    # Property characteristics
    land_sqft=10000,
    living_area=2500,
    age=10,
    structure_quality=4.5,
    SPEC_FEAT_VAL=15000,
    
    # Distances
    rail_dist=8000,
    ocean_dist=60000,
    water_dist=3000,
    center_dist=20000,
    subcenter_dist=12000,
    highway_dist=5000,
    
    # Zoning question
    zoning_question="What size tree requires a permit to be removed?"
)

agent.print_report(results)

### Example 3 - Complete Analysis with Image

In [ ]:
print("\n" + "🏠 EXAMPLE 3: COMPLETE ANALYSIS WITH PROPERTY IMAGE" + "\n")

results = agent.analyze_property(
    # Location
    latitude=28.55,
    longitude=-81.35,
    
    # Property characteristics
    land_sqft=7000,
    living_area=1800,
    age=20,
    structure_quality=3.5,
    SPEC_FEAT_VAL=5000,
    
    # Distances
    rail_dist=10000,
    ocean_dist=65000,
    water_dist=4000,
    center_dist=25000,
    subcenter_dist=10000,
    highway_dist=4000,
    
    # Image path (update with your actual image)
    image_path="../data/prop-damage-images-resized-512/property_images_r3_c5_processed_by_imagy.png",
    
    # Zoning question
    zoning_question="Are there any restrictions on home additions or renovations?"
)

agent.print_report(results)